In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, optimizers

print("TensorFlow version:", tf.__version__)


I0000 00:00:1784960018.337972   17136 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784960018.389097   17136 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784960020.133420   17136 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


In [2]:
BASE_DIR = Path("..").resolve()

# This should point to: D:\HealthAI-Project\datasets\CheXpertSmall
DATA_ROOT = BASE_DIR / "datasets"

# CheXpert folder inside it
CHEXPERT_DIR = DATA_ROOT / "CheXpert-v1.0-small"

TRAIN_CSV = CHEXPERT_DIR / "train.csv"
VALID_CSV = CHEXPERT_DIR / "valid.csv"

print("BASE_DIR:", BASE_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("CHEXPERT_DIR:", CHEXPERT_DIR)
print("TRAIN_CSV exists:", TRAIN_CSV.exists())
print("VALID_CSV exists:", VALID_CSV.exists())

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

print("Raw train shape:", train_df.shape)
print("Raw valid shape:", valid_df.shape)
print(train_df.head())

BASE_DIR: /home/chandan/Internship/HealthAI-Project
DATA_ROOT: /home/chandan/Internship/HealthAI-Project/datasets
CHEXPERT_DIR: /home/chandan/Internship/HealthAI-Project/datasets/CheXpert-v1.0-small
TRAIN_CSV exists: True
VALID_CSV exists: True
Raw train shape: (223414, 19)
Raw valid shape: (234, 19)
                                                Path     Sex  Age  \
0  CheXpert-v1.0-small/train/patient00001/study1/...  Female   68   
1  CheXpert-v1.0-small/train/patient00002/study2/...  Female   87   
2  CheXpert-v1.0-small/train/patient00002/study1/...  Female   83   
3  CheXpert-v1.0-small/train/patient00002/study1/...  Female   83   
4  CheXpert-v1.0-small/train/patient00003/study1/...    Male   41   

  Frontal/Lateral AP/PA  No Finding  Enlarged Cardiomediastinum  Cardiomegaly  \
0         Frontal    AP         1.0                         NaN           NaN   
1         Frontal    AP         NaN                         NaN          -1.0   
2         Frontal    AP         NaN     

In [3]:
print(train_df["Path"].head(3))

first_rel = train_df["Path"].iloc[0]
full_path = DATA_ROOT / first_rel

print("Relative:", first_rel)
print("Full path:", full_path)
print("File exists?", full_path.exists())


0    CheXpert-v1.0-small/train/patient00001/study1/...
1    CheXpert-v1.0-small/train/patient00002/study2/...
2    CheXpert-v1.0-small/train/patient00002/study1/...
Name: Path, dtype: str
Relative: CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg
Full path: /home/chandan/Internship/HealthAI-Project/datasets/CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg
File exists? True


In [4]:
DISEASES = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
    "Pneumonia",
    "Pneumothorax",
    "No Finding"
]

# Keep only needed columns
train_df = train_df[["Path"] + DISEASES].copy()
valid_df = valid_df[["Path"] + DISEASES].copy()

def process_labels(df, disease_cols):
    df = df.copy()
    for col in disease_cols:
        df[col] = df[col].fillna(0)
        df[col] = df[col].replace(-1, 1)  # uncertain -> positive
    return df

train_df = process_labels(train_df, DISEASES)
valid_df = process_labels(valid_df, DISEASES)

# Build full file paths using DATA_ROOT now
train_df["filepath"] = train_df["Path"].apply(lambda p: str(DATA_ROOT / p))
valid_df["filepath"] = valid_df["Path"].apply(lambda p: str(DATA_ROOT / p))

# Drop missing files (now should keep MANY)
train_df = train_df[train_df["filepath"].apply(os.path.exists)].reset_index(drop=True)
valid_df = valid_df[valid_df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print("Train images:", len(train_df))
print("Valid images:", len(valid_df))
train_df[["Path", "filepath"] + DISEASES].head()


Train images: 223414
Valid images: 234


,Path,filepath,Atelectasis,Cardiomegaly,Consolidation,Edema,Pleural Effusion,Pneumonia,Pneumothorax,No Finding
0,CheXpert-v1.0-small/train/patient00001/study1/...,/home/chandan/Internship/HealthAI-Project/data...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,/home/chandan/Internship/HealthAI-Project/data...,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,CheXpert-v1.0-small/train/patient00002/study1/...,/home/chandan/Internship/HealthAI-Project/data...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,CheXpert-v1.0-small/train/patient00002/study1/...,/home/chandan/Internship/HealthAI-Project/data...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,CheXpert-v1.0-small/train/patient00003/study1/...,/home/chandan/Internship/HealthAI-Project/data...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [5]:
train_df = train_df.sample(min(25000, len(train_df)), random_state=42)
valid_df = valid_df.sample(min(5000, len(valid_df)), random_state=42)

print("After subsample - Train:", len(train_df), "Valid:", len(valid_df))


After subsample - Train: 25000 Valid: 234


In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    horizontal_flip=True,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
)

valid_datagen = ImageDataGenerator(
    rescale=1.0/255.0
)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col="filepath",
    y_col=DISEASES,              # list of label columns
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",            # multi-label
    shuffle=True
)

val_gen = valid_datagen.flow_from_dataframe(
    dataframe=valid_df,
    x_col="filepath",
    y_col=DISEASES,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    shuffle=False
)


Found 25000 validated image filenames.
Found 234 validated image filenames.


In [7]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, optimizers
import tensorflow as tf

base_model = DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)

base_model.trainable = False  # first stage: freeze base

model_md = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(len(DISEASES), activation="sigmoid")  # one prob per disease
])

model_md.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",  # multi-label loss
    metrics=["accuracy"]
)

model_md.summary()


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 101s 3us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         8,200 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,045,704 (26.88 MB)

 Trainable params: 8,200 (32.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from pathlib import Path

MODELS_DIR = (BASE_DIR / "models")
MODELS_DIR.mkdir(exist_ok=True)

md_model_path = MODELS_DIR / "xray_chexpert_multidisease_model.keras"

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    md_model_path,
    monitor="val_loss",
    save_best_only=True
)

EPOCHS = 5  # start small – we can fine-tune more later

history_md = model_md.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)


Epoch 1/5


I0000 00:00:1784960130.414368   17136 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


782/782 ━━━━━━━━━━━━━━━━━━━━ 2496s 3s/step - accuracy: 0.2085 - loss: 0.4793 - val_accuracy: 0.1581 - val_loss: 0.4091
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 5723s 7s/step - accuracy: 0.2176 - loss: 0.4541 - val_accuracy: 0.1197 - val_loss: 0.4112
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2139s 3s/step - accuracy: 0.2238 - loss: 0.4513 - val_accuracy: 0.1538 - val_loss: 0.3989
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3305s 4s/step - accuracy: 0.2208 - loss: 0.4510 - val_accuracy: 0.2179 - val_loss: 0.3963
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3414s 4s/step - accuracy: 0.2192 - loss: 0.4498 - val_accuracy: 0.1154 - val_loss: 0.4047


In [9]:
val_loss, val_acc = model_md.evaluate(val_gen)
print("Multi-disease Val loss:", val_loss)
print("Multi-disease Val accuracy:", val_acc)


8/8 ━━━━━━━━━━━━━━━━━━━━ 29s 3s/step - accuracy: 0.2179 - loss: 0.3963
Multi-disease Val loss: 0.39626821875572205
Multi-disease Val accuracy: 0.21794871985912323


In [10]:
import json

labels_path = MODELS_DIR / "xray_chexpert_labels.json"

with open(labels_path, "w") as f:
    json.dump(DISEASES, f, indent=2)

print("Saved labels to:", labels_path)
print("Saved model to:", md_model_path)


Saved labels to: /home/chandan/Internship/HealthAI-Project/models/xray_chexpert_labels.json
Saved model to: /home/chandan/Internship/HealthAI-Project/models/xray_chexpert_multidisease_model.keras


In [11]:
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np

def predict_chexpert_image(img_path, model, diseases, img_size=(224, 224)):
    img = load_img(img_path, target_size=img_size, color_mode="rgb")
    arr = img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = model.predict(arr)[0]  # shape: (len(diseases),)

    return {diseases[i]: float(probs[i]) for i in range(len(diseases))}


In [12]:
sample_path = valid_df["filepath"].iloc[0]
print("Sample image:", sample_path)

preds = predict_chexpert_image(sample_path, model_md, DISEASES, IMG_SIZE)
preds


Sample image: /home/chandan/Internship/HealthAI-Project/datasets/CheXpert-v1.0-small/valid/patient64592/study1/view1_frontal.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


{'Atelectasis': 0.42938241362571716,
 'Cardiomegaly': 0.1311318427324295,
 'Consolidation': 0.11708790063858032,
 'Edema': 0.13630300760269165,
 'Pleural Effusion': 0.4339037835597992,
 'Pneumonia': 0.059872549027204514,
 'Pneumothorax': 0.04845339432358742,
 'No Finding': 0.11006457358598709}